### Import Libraries

In [1]:
import pandas as pd
import numpy as np
print("import Done")

import Done


### Load Dataset

In [2]:
cs = pd.read_csv("dirty_cafe_sales.csv")
print("Read Done")

Read Done


### Data Profiling

In [3]:
cs.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [4]:
cs.isnull().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

### Data Cleaning

In [5]:
clean_cs = cs.copy()

In [6]:
clean_cs.isnull().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

In [7]:
clean_cs["Item"].unique()

<StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',  'UNKNOWN',
 'Sandwich',        nan,    'ERROR',    'Juice',      'Tea']
Length: 11, dtype: str

In [8]:
clean_cs["Item"].duplicated().sum()

np.int64(9989)

In [9]:
clean_cs = clean_cs.drop_duplicates()

In [10]:
clean_cs["Item"] = clean_cs["Item"].replace(["UNKNOWN", "ERROR"], np.nan)

In [11]:
clean_cs["Item"].unique()

<StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',        nan,
 'Sandwich',    'Juice',      'Tea']
Length: 9, dtype: str

In [12]:
clean_cs["Quantity"].unique()

<StringArray>
['2', '4', '5', '3', '1', 'ERROR', 'UNKNOWN', nan]
Length: 8, dtype: str

In [14]:
clean_cs.dtypes

Transaction ID      str
Item                str
Quantity            str
Price Per Unit      str
Total Spent         str
Payment Method      str
Location            str
Transaction Date    str
dtype: object

In [15]:
clean_cs["Quantity"] = pd.to_numeric(clean_cs["Quantity"], errors="coerce")

In [16]:
clean_cs["Price Per Unit"] = pd.to_numeric(clean_cs["Price Per Unit"], errors="coerce")

In [17]:
clean_cs["Total Spent"] = pd.to_numeric(clean_cs["Total Spent"], errors="coerce")

In [18]:
clean_cs[
    clean_cs["Quantity"].isnull() &
    clean_cs["Price Per Unit"].notnull() &
    clean_cs["Total Spent"].notnull()
].shape

(441, 8)

In [19]:
quantity_mask = (
    clean_cs["Quantity"].isnull() &
    clean_cs["Price Per Unit"].notnull() &
    clean_cs["Total Spent"].notnull()
)

In [20]:
clean_cs.dtypes

Transaction ID          str
Item                    str
Quantity            float64
Price Per Unit      float64
Total Spent         float64
Payment Method          str
Location                str
Transaction Date        str
dtype: object

In [21]:
clean_cs.loc[quantity_mask, "Quantity"] = (
    clean_cs.loc[quantity_mask, "Total Spent"] / clean_cs.loc[quantity_mask, "Price Per Unit"]
)

In [22]:
clean_cs.loc[quantity_mask, ["Quantity", "Price Per Unit", "Total Spent"]].head(10)

,Quantity,Price Per Unit,Total Spent
20,5.0,4.0,20.0
55,2.0,1.0,2.0
57,1.0,3.0,3.0
66,2.0,3.0,6.0
117,3.0,3.0,9.0
153,4.0,3.0,12.0
177,5.0,5.0,25.0
178,4.0,4.0,16.0
189,3.0,1.0,3.0
198,4.0,3.0,12.0


In [23]:
clean_cs["Quantity"].isnull().sum()

np.int64(38)

In [24]:
clean_cs[clean_cs["Quantity"].isnull()][["Transaction ID", "Item", "Quantity", "Price Per Unit", "Total Spent"]]

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent
236,TXN_8562645,Salad,NaN,5.0,NaN
278,TXN_3229409,Juice,NaN,3.0,NaN
629,TXN_9289174,Cake,NaN,NaN,12.0
641,TXN_2962976,Juice,NaN,3.0,NaN
738,TXN_8696094,Sandwich,NaN,4.0,NaN
912,TXN_1575608,Sandwich,NaN,NaN,20.0
1008,TXN_7225428,Tea,NaN,NaN,3.0
1436,TXN_7590801,Tea,NaN,NaN,6.0
1482,TXN_3593060,Smoothie,NaN,NaN,16.0
2330,TXN_3849488,Salad,NaN,NaN,5.0


In [25]:

clean_cs[clean_cs["Quantity"].isnull()][["Item", "Price Per Unit", "Total Spent"]].isnull().sum()

Item               3
Price Per Unit    18
Total Spent       20
dtype: int64

In [26]:
#this gives the median quantity for each item
clean_cs["Quantity"] = clean_cs["Quantity"].fillna(
    clean_cs.groupby("Item")["Quantity"].transform("median")
)

In [28]:
clean_cs["Quantity"] = clean_cs["Quantity"].fillna(clean_cs["Quantity"].mode()[0])

In [29]:
clean_cs["Quantity"].isnull().sum()

np.int64(0)

In [30]:
clean_cs["Price Per Unit"].isnull().sum()

np.int64(533)

In [31]:
clean_cs[
    clean_cs["Price Per Unit"].isnull() &
    clean_cs["Quantity"].notnull() &
    clean_cs["Total Spent"].notnull()
].shape

(513, 8)

In [32]:
price_mask = (
    clean_cs["Price Per Unit"].isnull() &
    clean_cs["Quantity"].notnull() &
    clean_cs["Total Spent"].notnull()
)

In [33]:
clean_cs.loc[price_mask, "Price Per Unit"] = (
    clean_cs.loc[price_mask, "Total Spent"] / clean_cs.loc[price_mask, "Quantity"]
)

In [34]:
clean_cs["Price Per Unit"].isnull().sum()

np.int64(20)

In [35]:
clean_cs["Price Per Unit"] = clean_cs["Price Per Unit"].fillna(
    clean_cs.groupby("Item")["Price Per Unit"].transform("median")
)

In [37]:
clean_cs["Price Per Unit"] = clean_cs["Price Per Unit"].fillna(clean_cs["Price Per Unit"].mode()[0])

In [38]:
clean_cs["Price Per Unit"].isnull().sum()

np.int64(0)

In [39]:
clean_cs["Total Spent"].isnull().sum()

np.int64(502)

In [40]:
total_mask = (
    clean_cs["Total Spent"].isnull() &
    clean_cs["Quantity"].notnull() &
    clean_cs["Price Per Unit"].notnull()
)

In [41]:
clean_cs.loc[total_mask, "Total Spent"] = (
    clean_cs.loc[total_mask, "Quantity"] * clean_cs.loc[total_mask, "Price Per Unit"]
)

In [42]:
clean_cs["Price Per Unit"].isnull().sum()

np.int64(0)

In [43]:
clean_cs["Payment Method"].unique()

<StringArray>
['Credit Card', 'Cash', 'UNKNOWN', 'Digital Wallet', 'ERROR', nan]
Length: 6, dtype: str

In [44]:
clean_cs["Payment Method"] = clean_cs["Payment Method"].replace(["UNKNOWN", "ERROR"], np.nan)

In [45]:
clean_cs["Payment Method"] = clean_cs["Payment Method"].fillna(clean_cs["Payment Method"].mode()[0])

In [46]:
clean_cs["Payment Method"].isnull().sum()

np.int64(0)

In [47]:
clean_cs["Payment Method"].unique()

<StringArray>
['Credit Card', 'Cash', 'Digital Wallet']
Length: 3, dtype: str

In [48]:
clean_cs["Location"].isnull().sum()

np.int64(3265)

In [49]:
clean_cs["Location"].unique()

<StringArray>
['Takeaway', 'In-store', 'UNKNOWN', nan, 'ERROR']
Length: 5, dtype: str

In [50]:
clean_cs["Location"].value_counts()

Location
Takeaway    3022
In-store    3017
ERROR        358
UNKNOWN      338
Name: count, dtype: int64

In [51]:
clean_cs["Location"] = clean_cs["Location"].replace({"UNKNOWN":"Takeaway", "ERROR": "In-store"})

In [52]:
clean_cs["Location"].value_counts()

Location
In-store    3375
Takeaway    3360
Name: count, dtype: int64

In [53]:
clean_cs["Location"].isnull().sum()

np.int64(3265)

In [54]:
clean_cs["Location"] = clean_cs["Location"].fillna(clean_cs["Location"].mode()[0])

In [55]:
clean_cs["Location"].isnull().sum()

np.int64(0)

In [56]:
clean_cs["Location"].unique()

<StringArray>
['Takeaway', 'In-store']
Length: 2, dtype: str

In [57]:
clean_cs["Transaction Date"] = pd.to_datetime(clean_cs["Transaction Date"], errors="coerce")

In [58]:
clean_cs ["Transaction Date"] = clean_cs["Transaction Date"].fillna(clean_cs["Transaction Date"].interpolate())

In [59]:
clean_cs ["Transaction Date"].isnull().sum()

np.int64(0)

In [78]:
clean_cs["Transaction Date"] = pd.to_datetime(clean_cs["Transaction Date"]).dt.date

In [60]:
clean_cs["Item"].unique()

<StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',        nan,
 'Sandwich',    'Juice',      'Tea']
Length: 9, dtype: str

In [61]:
clean_cs.groupby("Item")["Price Per Unit"].unique()

Item
Cake                                      [3.0, 4.0, 1.0]
Coffee                          [2.0, 1.3333333333333333]
Cookie      [1.0, 0.3333333333333333, 0.6666666666666666]
Juice                                               [3.0]
Salad                           [5.0, 1.6666666666666667]
Sandwich                         [4.0, 6.666666666666667]
Smoothie     [4.0, 5.333333333333333, 2.6666666666666665]
Tea                                  [1.5, 1.0, 2.0, 2.5]
Name: Price Per Unit, dtype: object

In [62]:
item_value = (clean_cs.dropna(subset=["Item"])
             .groupby("Price Per Unit")["Item"]
             .agg(lambda x: x.mode()[0]))

In [63]:
clean_cs["Item"] = clean_cs["Item"].fillna(clean_cs["Price Per Unit"].map(item_value))

In [66]:
clean_cs["Item"] = clean_cs["Item"].fillna(clean_cs["Item"].mode()[0])

In [67]:
clean_cs["Item"].isnull().sum()

np.int64(0)

In [69]:
clean_cs.isnull().sum()

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

In [70]:
clean_cs["Item"].unique()

<StringArray>
['Coffee', 'Cake', 'Cookie', 'Salad', 'Smoothie', 'Juice', 'Sandwich', 'Tea']
Length: 8, dtype: str

In [72]:
clean_cs["Payment Method"].unique()

<StringArray>
['Credit Card', 'Cash', 'Digital Wallet']
Length: 3, dtype: str

In [73]:
clean_cs["Location"].unique()

<StringArray>
['Takeaway', 'In-store']
Length: 2, dtype: str

In [74]:
print("Original Rows:", len(cs))
print("Cleaned Rows:", len(clean_cs))

Original Rows: 10000
Cleaned Rows: 10000


In [75]:
print("Original missing Values:", cs.isnull().sum())
print("Remaining missing Values:", clean_cs.isnull().sum())

Original missing Values: Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64
Remaining missing Values: Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64


### Export Clean Data

In [79]:
clean_cs.to_csv("Cleaned_Cafe_Sales.csv", index=False)
print("Export Done")

Export Done
